# File and Project Overview

This notebook belongs to the `phase1` implementation of the Fake Review Detection System.

Project goal:
- Predict whether a review is `fake` or `genuine`
- Return a `fake probability` score for each review

About this file (`phase1_v2_colab.ipynb`):
- This is the **V2 text-focused model** notebook
- It reduces metadata bias by excluding behavioral features from training
- It includes full flow: install deps, load data, clean/split, feature build, train/evaluate, save artifacts, and sample prediction

Dataset used in this phase:
- `dataset/amazon_labeled_fake_reviews/final_labeled_fake_reviews.csv`

How to run:
- Execute cells from top to bottom
- Keep output artifact paths as-is for Colab, or change if needed for local runs

# Phase 1 v2 - Fake Review Detection (Colab)

This notebook is a complete v2 (text-focused) training pipeline in one place.

Run cells top-to-bottom.

In [ ]:
# Install required packages (safe to re-run).
!pip -q install pandas numpy scikit-learn nltk scipy joblib

In [ ]:
# Imports and notebook-level config.
import os
import re
import json
import time
import hashlib
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, brier_score_loss
from scipy.sparse import csr_matrix, hstack

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Imports loaded.')

In [ ]:
# Load dataset (local path first, then Colab upload fallback).
LOCAL_DATA_PATH = 'dataset/amazon_labeled_fake_reviews/final_labeled_fake_reviews.csv'
COLAB_DATA_PATH = '/content/final_labeled_fake_reviews.csv'

if os.path.exists(LOCAL_DATA_PATH):
    DATA_PATH = LOCAL_DATA_PATH
elif os.path.exists(COLAB_DATA_PATH):
    DATA_PATH = COLAB_DATA_PATH
else:
    try:
        from google.colab import files
        print('Upload final_labeled_fake_reviews.csv ...')
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise ValueError('No file uploaded')
        DATA_PATH = '/content/' + list(uploaded.keys())[0]
    except Exception as ex:
        raise RuntimeError('Set DATA_PATH manually if local/Colab auto-detect fails.') from ex

df_raw = pd.read_csv(DATA_PATH)
print('Using DATA_PATH:', DATA_PATH)
print('Rows:', len(df_raw))
print('Columns:', list(df_raw.columns))

In [ ]:
# Cleaning, dedupe, and leakage-safe split helpers.
def normalize_label(value):
    # Normalize labels to integer 0/1.
    if pd.isna(value):
        return np.nan
    try:
        return int(value)
    except Exception:
        value_str = str(value).strip().lower()
        if value_str in {'fake', '1', 'true', 'yes', 'y'}:
            return 1
        if value_str in {'genuine', 'real', '0', 'false', 'no', 'n'}:
            return 0
        return np.nan

def basic_text_clean(text):
    # Normalize text for dedupe/split hashing.
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def make_text_hash(text_value):
    # Build deterministic group hash.
    return hashlib.md5(text_value.encode('utf-8')).hexdigest()

df = pd.DataFrame()
df['text'] = df_raw['text'].astype(str)
df['label'] = df_raw['label'].apply(normalize_label)
df['rating'] = pd.to_numeric(df_raw.get('rating'), errors='coerce').fillna(0.0)
df['helpful_vote'] = pd.to_numeric(df_raw.get('helpful_vote'), errors='coerce').fillna(0.0)
df['verified_purchase'] = df_raw.get('verified_purchase', pd.Series(['FALSE'] * len(df_raw))).astype(str).str.upper().map({'TRUE':1,'FALSE':0}).fillna(0).astype(int)
df['text_clean_for_split'] = df['text'].apply(basic_text_clean)

# Remove invalid rows.
df = df[df['label'].isin([0,1])]
df = df[df['text_clean_for_split'].str.len() > 0]

# Dedupe before split to reduce leakage.
df = df.drop_duplicates(subset=['text_clean_for_split']).reset_index(drop=True)
df['text_hash'] = df['text_clean_for_split'].apply(make_text_hash)

group_df = df.groupby('text_hash', as_index=False)['label'].first()
keys = group_df['text_hash'].values
y_group = group_df['label'].values

# Split hash groups: train/calibration/validation/test.
train_calib_val_keys, test_keys, y_train_calib_val, _ = train_test_split(
    keys, y_group, test_size=0.10, stratify=y_group, random_state=RANDOM_SEED
)

train_val_keys, calib_keys, y_train_val, _ = train_test_split(
    train_calib_val_keys,
    y_train_calib_val,
    test_size=(0.10 / 0.90),
    stratify=y_train_calib_val,
    random_state=RANDOM_SEED
)

train_keys, val_keys, _, _ = train_test_split(
    train_val_keys,
    y_train_val,
    test_size=(0.10 / 0.80),
    stratify=y_train_val,
    random_state=RANDOM_SEED
)

split_map = {
    'train': set(train_keys.tolist()),
    'calibration': set(calib_keys.tolist()),
    'validation': set(val_keys.tolist()),
    'test': set(test_keys.tolist())
}

df['split'] = 'unassigned'
for split_name, key_set in split_map.items():
    df.loc[df['text_hash'].isin(key_set), 'split'] = split_name

assert (df['split'] != 'unassigned').all()
print(df['split'].value_counts())
print('Label mean by split:\n', df.groupby('split')['label'].mean())

In [ ]:
# Feature engineering for v2 (text + text-style numeric, no behavioral features).
def clean_text_for_model(text):
    # Normalize text for TF-IDF.
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r"[^a-z0-9\s!?']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def punctuation_ratio(text):
    # Compute punctuation density.
    if not text:
        return 0.0
    punct_count = sum(1 for c in text if c in '!?.,:;\"\'()-[]{}')
    return punct_count / max(len(text), 1)

def uppercase_ratio(text):
    # Compute uppercase ratio.
    if not text:
        return 0.0
    upper = sum(1 for c in text if c.isupper())
    alpha = sum(1 for c in text if c.isalpha())
    return upper / max(alpha, 1)

def build_numeric_features(frame):
    # Build text-style numeric feature set used by v2.
    out = pd.DataFrame(index=frame.index)
    raw_text = frame['text'].fillna('').astype(str)
    clean = frame['text_model'].fillna('').astype(str)

    out['char_count'] = raw_text.str.len()
    out['word_count'] = clean.str.split().str.len().fillna(0)
    out['avg_word_len'] = out['char_count'] / np.maximum(out['word_count'], 1)
    out['exclamation_count'] = raw_text.str.count('!')
    out['question_count'] = raw_text.str.count(r'\?')
    out['punctuation_ratio'] = raw_text.apply(punctuation_ratio)
    out['uppercase_ratio'] = raw_text.apply(uppercase_ratio)
    return out.astype(float)

split_frames = {k: v.copy() for k, v in df.groupby('split')}
for k in split_frames:
    split_frames[k]['text_model'] = split_frames[k]['text'].apply(clean_text_for_model)

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=0.98,
    max_features=50000,
    sublinear_tf=True
)
vectorizer.fit(split_frames['train']['text_model'])

def make_matrix(frame):
    # Build sparse matrix for one split frame.
    x_text = vectorizer.transform(frame['text_model'])
    x_num = build_numeric_features(frame)
    return x_text, x_num

x_train_text, train_num = make_matrix(split_frames['train'])
x_cal_text, cal_num = make_matrix(split_frames['calibration'])
x_val_text, val_num = make_matrix(split_frames['validation'])
x_test_text, test_num = make_matrix(split_frames['test'])

num_mean = train_num.mean(axis=0)
num_std = train_num.std(axis=0).replace(0, 1.0)

def scale_num(num):
    # Scale numeric features with train statistics.
    return (num - num_mean) / num_std

x_train = hstack([x_train_text, csr_matrix(scale_num(train_num).values)], format='csr')
x_cal = hstack([x_cal_text, csr_matrix(scale_num(cal_num).values)], format='csr')
x_val = hstack([x_val_text, csr_matrix(scale_num(val_num).values)], format='csr')
x_test = hstack([x_test_text, csr_matrix(scale_num(test_num).values)], format='csr')

y_train = split_frames['train']['label'].astype(int).values
y_cal = split_frames['calibration']['label'].astype(int).values
y_val = split_frames['validation']['label'].astype(int).values
y_test = split_frames['test']['label'].astype(int).values

print('Train matrix shape:', x_train.shape)

In [ ]:
# Train candidate models and select best by validation F1.
def expected_calibration_error(y_true, y_prob, bins=10):
    # Compute ECE from probability bins.
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for i in range(bins):
        lo, hi = edges[i], edges[i+1]
        mask = (y_prob >= lo) & (y_prob < hi if i < bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def pick_best_threshold(y_true, y_prob):
    # Pick threshold maximizing validation F1.
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.05, 0.95, 19):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1

def eval_metrics(y_true, y_prob, threshold):
    # Compute summary metrics for one split.
    pred = (y_prob >= threshold).astype(int)
    return {
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'brier': float(brier_score_loss(y_true, y_prob)),
        'ece_10': float(expected_calibration_error(y_true, y_prob, bins=10)),
        'confusion_matrix': confusion_matrix(y_true, pred).tolist()
    }

models = {}

# Logistic baseline.
log_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)
log_model.fit(x_train, y_train)
models['logistic_regression'] = log_model

# SGD with probabilistic loss.
sgd_model = SGDClassifier(loss='log_loss', alpha=1e-5, max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)
sgd_model.fit(x_train, y_train)
models['sgd_log_loss'] = sgd_model

# LinearSVC + calibration on calibration split.
svc_base = LinearSVC(class_weight='balanced', random_state=RANDOM_SEED, max_iter=5000)
svc_base.fit(x_train, y_train)
svc_cal = CalibratedClassifierCV(svc_base, method='sigmoid', cv=3)
svc_cal.fit(x_cal, y_cal)
models['linear_svc_calibrated'] = svc_cal

ranked = []
for name, model in models.items():
    # Evaluate each candidate on validation split.
    prob = model.predict_proba(x_val)[:, 1] if hasattr(model, 'predict_proba') else (1.0 / (1.0 + np.exp(-model.decision_function(x_val))))
    t, _ = pick_best_threshold(y_val, prob)
    m = eval_metrics(y_val, prob, t)
    ranked.append({'name': name, 'threshold': t, 'metrics': m})

ranked = sorted(ranked, key=lambda x: x['metrics']['f1'], reverse=True)
best = ranked[0]
best_name = best['name']
best_t = best['threshold']
best_model = models[best_name]

test_prob = best_model.predict_proba(x_test)[:, 1] if hasattr(best_model, 'predict_proba') else (1.0 / (1.0 + np.exp(-best_model.decision_function(x_test))))
test_metrics = eval_metrics(y_test, test_prob, best_t)

print('Selected model:', best_name)
print('Validation F1:', best['metrics']['f1'])
print('Test F1:', test_metrics['f1'])
print('Test metrics:', test_metrics)

In [ ]:
# Save artifacts and run sample predictions.
OUT_DIR = '/content/phase1_v2_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(best_model, os.path.join(OUT_DIR, 'best_model.joblib'))
joblib.dump(vectorizer, os.path.join(OUT_DIR, 'tfidf_vectorizer.joblib'))

feature_metadata = {
    'numeric_columns': list(train_num.columns),
    'numeric_mean': num_mean.to_dict(),
    'numeric_std': num_std.to_dict(),
    'include_behavioral': False
}
with open(os.path.join(OUT_DIR, 'feature_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(feature_metadata, f, indent=2)

model_metadata = {
    'model_name': best_name,
    'model_version': 'phase1-v2-colab',
    'threshold': float(best_t),
    'random_seed': int(RANDOM_SEED),
    'include_behavioral': False
}
with open(os.path.join(OUT_DIR, 'model_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2)

def predict_one(text, rating=0, helpful_vote=0, verified_purchase=0):
    # Predict fake probability for a single review using text-focused features.
    frame = pd.DataFrame([{
        'text': text,
        'rating': rating,
        'helpful_vote': helpful_vote,
        'verified_purchase': verified_purchase
    }])
    frame['text_model'] = frame['text'].apply(clean_text_for_model)
    x_text = vectorizer.transform(frame['text_model'])
    x_num = build_numeric_features(frame)
    x_num_scaled = scale_num(x_num)
    x_mat = hstack([x_text, csr_matrix(x_num_scaled.values)], format='csr')
    if hasattr(best_model, 'predict_proba'):
        prob = float(best_model.predict_proba(x_mat)[:, 1][0])
    else:
        score = float(best_model.decision_function(x_mat)[0])
        prob = float(1.0 / (1.0 + np.exp(-score)))
    label = 'fake' if prob >= best_t else 'genuine'
    return {
        'label': label,
        'fake_probability': prob,
        'fake_percent': round(prob * 100, 2),
        'threshold': best_t
    }

example = predict_one(
    text='I used this for two weeks. Build quality is decent and setup was easy, but battery lasts around 5 hours instead of the 7 listed.',
    rating=3,
    helpful_vote=2,
    verified_purchase=1
)
print('Example prediction:', example)
print('Artifacts saved in:', OUT_DIR)